# 0 - IMPORTS

In [1]:
import glob
import fitparse
import numpy as np
import pandas as pd
import plotly as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px

## 0.1 - Helper Functions

In [2]:
def fit_to_dataframe(file_path):
    fitfile = fitparse.FitFile(file_path)
    records = []
    for record in fitfile.get_messages('record'):
        data = {}
        for field in record:
            data[field.name] = field.value
        records.append(data)
    return pd.DataFrame(records)


def clean_session(df):
    " "
    
    # 1. Filtra colunas relevantes
    df = df[['distance', 'timestamp', 'cadence', 'heart_rate', 'speed']]
    # 2. Corrige delay do relogio apos pausa, onde distancia mudou mas speed continuou 0 
    mask = (df['speed'] == 0) & (df['distance'].diff() > 0)
    df.loc[mask, 'speed'] = np.nan
    # 3. Interpola NaNs no meio
    df = df.interpolate(method='linear')
    # 4. Corrige cadencia diferente de 0 para quando eu estava parado
    df.loc[(df['speed'] == 0) & (df['cadence'] > 0), 'cadence'] = 0
    # 5. Remove linhas iniciais ainda com NaN ou distância 0
    df = df.dropna(subset=['distance', 'timestamp', 'cadence', 'heart_rate', 'speed'])
    df = df[df['distance'] > 0]
    # 6. Reset index
    df = df.reset_index(drop=True)
    return df


def get_zone(hr):
    if hr <= 134:
        return 'Z1'
    elif hr <= 146:
        return 'Z2'
    elif hr <= 155:
        return 'Z3'
    elif hr <= 165:
        return 'Z4'
    else:
        return 'Z5'

def seconds_to_mmss(decimal_minutes):
    if decimal_minutes == 0:
        return '00:00'
    total_seconds = int(decimal_minutes * 60)
    minutes = total_seconds // 60
    seconds = total_seconds % 60
    return f"{minutes:02d}:{seconds:02d}"

def feature_engineering(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['year']  = df['timestamp'].dt.year
    df['month'] = df['timestamp'].dt.month
    df['day']   = df['timestamp'].dt.day
    df['week']  = df['timestamp'].dt.isocalendar().week
    df['time']  = df['timestamp'].dt.time
    df['pace'] = df['speed'].apply(lambda x: 0 if x == 0 else round(1000 / (x * 60), 2))
    df['pace_m'] = df['pace'].apply(seconds_to_mmss)
    df['spm']   = df['cadence'] * 2
    df['zone']  = df['heart_rate'].apply(get_zone)
    df['session'] = 0
    df['training_type'] = 0
    df = df.drop(columns=['timestamp'])
    return df

# 1 - DATA MANIPULATION

## 1.1 - Loading Data

In [3]:
files = glob.glob('../data/raw/*.fit')
files.sort()

sessions = []
for file in files:
    df = fit_to_dataframe(file)
    df = clean_session(df)
    df = feature_engineering(df)
    sessions.append(df)

In [4]:
final_df = pd.concat(sessions, ignore_index=True)

# 2 - FEATURE ENGINEERING

In [5]:
# Cria coluna date
final_df['date'] = pd.to_datetime(final_df[['year', 'month', 'day']])

# Numera as sessões por ordem cronológica
final_df['session'] = final_df['date'].rank(method='dense').astype(int)

# Training type fixo
final_df['training_type'] = 'easy_run'

# Remove coluna auxiliar
final_df = final_df.drop(columns=['date'])

# Duracao do treino
final_df['ts_duration'] = final_df.groupby('session').cumcount() + 1
final_df['ts_duration'] = pd.to_timedelta(final_df['ts_duration'], unit='s').apply(lambda x: str(x).split(' ')[-1])


# 3 - SAVING CLEANED DATASET

In [6]:
final_df.to_csv('../data/processed/treinos.csv', index=False)

# 4 - EXPLORATORY DATA ANALYSIS

- Pace Min, Max e Avg por session
- FC Min,Max e Avg por session
- Linha do tempo mostrando FC e Pace ao longo do tempo.

In [ ]:
df1 = final_df.copy()

In [ ]:
df1.dtypes

In [ ]:
df1.head()

In [ ]:
df1[['session','training_type','pace','heart_rate','spm']].groupby(['training_type','session']).mean().round(2).reset_index()

In [ ]:
#### Plotando variacao da FC ao longo de um treino
df_s1 = df1[df1['session'] == 18]
df_s1.tail()

In [ ]:
avg_pace    = df_s1['pace'].mean().round(2)
tt_distance = (df_s1['distance'].iloc[-1] / 1000).round(2)
tt_time     = df_s1['ts_duration'].iloc[-1]
avg_spm     = df_s1['spm'].mean().round(0)
print('Avg. Pace: {}'.format( avg_pace ) )
print('Distance: {}'.format( tt_distance ) )
print('Durtation: {}'.format( tt_time ) )
print('Cadence: {}'.format( avg_spm ) )

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=df_s1['ts_duration'],
    y=df_s1['heart_rate'],
    mode='lines',
    line=dict(color='red', width=2),
    name='FC'
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_s1['ts_duration'],
    y=df_s1['pace'],
    mode='lines',
    line=dict(color='blue', width=2),
    name='Pace'
), secondary_y=True)

fig.update_layout(title='FC e Pace ao longo do treino')
fig.update_yaxes(title_text='BPM', secondary_y=False)
fig.update_yaxes(title_text='min/km', sebcondary_y=True, autorange='reversed')

fig.show()

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=df_s1['ts_duration'],
    y=df_s1['heart_rate'],
    mode='lines',
    line=dict(color='green', width=1),
    name='FC'
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_s1['ts_duration'],
    y=df_s1['pace'],
    mode='lines',
    line=dict(color='black', width=1),
    name='Pace'
), secondary_y=True)

fig.update_layout(
    title='FC e Pace ao longo do treino',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
)

fig.update_yaxes(title_text='BPM', secondary_y=False)
fig.update_yaxes(title_text='min/km', secondary_y=True, autorange='reversed')

fig.show()

In [ ]:
# Tempo em cada FC Zone

tempo_por_zona = df_s1.groupby('zone').size().reset_index(name='segundos')

tempo_por_zona['minutos'] = tempo_por_zona['segundos'] / 60  # numérico pro plotly

tempo_por_zona['label'] = tempo_por_zona['segundos'].apply(
    lambda s: f"{int((s % 3600) // 60):02d}:{int(s % 60):02d}"
)

fig = px.bar(
    tempo_por_zona, 
    x='minutos',  # numérico
    y='zone', 
    orientation='h',
    text='label'  # exibe MM:SS nas barras
)

fig.update_traces(textposition='outside')
fig.show()

In [ ]:
tempo_por_zona = df_s1.groupby('zone').size().reset_index(name='segundos')
tempo_por_zona['minutos'] = tempo_por_zona['segundos'] / 60
tempo_por_zona['label'] = tempo_por_zona['segundos'].apply(
    lambda s: f"{int((s % 3600) // 60):02d}:{int(s % 60):02d}"
)

cores = {
    'Z1': 'blue',
    'Z2': 'green',
    'Z3': 'yellow',
    'Z4': 'orange',
    'Z5': 'red'
}

fig = px.bar(
    tempo_por_zona,
    x='minutos',
    y='zone',
    orientation='h',
    text='label',
    color='zone',
    color_discrete_map=cores,
    title='⏱️ Tempo por Zona de FC'
)

fig.update_traces(textposition='outside', width=0.4)  # width = espessura das barras

fig.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    showlegend=False
)

fig.show()